# EmpowerLens - Experiments 1-8, re-run so the results are comparable

Replaces the previous version of this notebook. Full reasoning in
[`docs/RERUN_PLAN.md`](../docs/RERUN_PLAN.md); the short version is below.

## Why the old results could not be used

| # | what was wrong | measured |
|---|---|---|
| 1 | The training data contained the test set | **396 rows** of `data/splits_combined/train.csv` also appear in Annotated val/test - **195 of 253 test rows, 77%** |
| 2 | Those leaked rows mostly carry the *wrong* label | the two corpora agree on only **36%** of them, so the model saw the test text with the wrong answer |
| 3 | Duplicate rows silently re-weighted training | 4,645 train rows, only **3,019 unique texts** |
| 4 | Every experiment sat a different exam | E2 scored each corpus on its own test set, E3-E8 on Combined, Month-1 on Annotated |
| 5 | Two copies of the script, wrong branch, scattered output | `src/` and `experiments/` both held it; notebook cloned `lumia-space`; results split across `results/` and `result_experiment/` |

## What is different now

- **Clean splits.** `src.make_splits_clean` writes *new* dirs, dropping any train
  row that appears in the yardstick's val/test plus internal duplicates. The
  frozen dirs are never edited, so old results stay traceable.
- **Two exams per run.** Every checkpoint is scored on its **home** test set
  (within-dataset performance) *and* on the **yardstick** -
  `data/splits/test.csv`, the same 253 human-annotated rows for every
  experiment. `transfer_gap = home - yardstick`.
- **One results root**, `results_rerun/`, one naming scheme, one branch.
- **Same protocol everywhere** - same model, same 3 seeds, same epochs. Only the
  thing under test varies.

> **The rule this notebook enforces:** two numbers are comparable only if both
> come from the **yardstick** exam, on the **same task**, with `leaked = False`.
> Every results row carries all three fields.

## 0. Setup

**Kaggle:** Settings -> Accelerator **GPU T4**, Internet **On**.

`CUDA_VISIBLE_DEVICES=0` pins to one GPU - T4 x2 causes a cross-device deadlock
in this stack (documented in the project notes).

> ⚠️ **Run this cell exactly once per session.** It starts with `rm -rf`, so
> re-running it deletes every result and checkpoint trained so far. If you need
> to restart, begin from the next cell.

In [ ]:
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import sys, json, subprocess
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "experiments/experiments_flat_mentalroberta.py").exists():
    if (ROOT.parent / "experiments/experiments_flat_mentalroberta.py").exists():
        ROOT = ROOT.parent
        os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

SCRIPT = "experiments/experiments_flat_mentalroberta.py"
assert Path(SCRIPT).exists(), f"{SCRIPT} not found - is the branch right?"
PY = sys.executable

# ---- one place to change the protocol; every experiment below uses it ----
MODEL     = "mental/mental-roberta-base"
SEEDS     = "42,1337,2024"        # project convention: 3 seeds, mean +/- std
EPOCHS    = 4
MAX_LEN   = 512
OUT_ROOT  = "results_rerun"       # ONE root, not results/ + result_experiment/

YARDSTICK = "data/splits"                    # the fixed exam, never changes
ANNOTATED = "data/splits"
CODIPAS   = "data/splits_codipas_clean"      # created by the preflight cell
COMBINED  = "data/splits_combined_clean"     # created by the preflight cell

print(f"repo   : {ROOT}")
print(f"script : {SCRIPT}")
print(f"model  : {MODEL} | seeds {SEEDS} | {EPOCHS} epochs | max_len {MAX_LEN}")
print(f"out    : {OUT_ROOT}/")


def run(cmd, label=""):
    # Stream a subprocess live so a long training run shows progress.
    if label:
        print(f"\n{'=' * 72}\n{label}\n{'=' * 72}")
    print("$", " ".join(str(c) for c in cmd), "\n")
    p = subprocess.Popen([str(c) for c in cmd], cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise SystemExit(f"FAILED (exit {p.returncode}): {' '.join(map(str, cmd))}")
    return p.returncode

## 1. Preflight - prove there is no leakage

**Nothing below this cell is worth running until this passes.** It rebuilds the
derived split dirs with the contamination removed, then re-audits its own output
and fails if anything survives.

`data/splits` (the yardstick) is never touched - it is the reference, and it has
no leak.

In [ ]:
# Build the clean dirs. --force is safe here: they are derived, not frozen,
# and fully reproducible from the sources plus this script.
run([PY, "-m", "src.make_splits_clean", "--force"], "Building leakage-free splits")

# Now prove it. Exit code 1 means a leak survived somewhere.
print(f"\n{'=' * 72}\nAUDIT\n{'=' * 72}")
p = subprocess.run([PY, "-m", "src.make_splits_clean", "--check"], cwd=ROOT,
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(p.stdout)

for d in (ANNOTATED, CODIPAS, COMBINED):
    man = Path(d) / "clean_manifest.json"
    n = len(pd.read_csv(Path(d) / "train.csv", encoding="utf-8-sig"))
    note = ""
    if man.exists():
        m = json.loads(man.read_text(encoding="utf-8"))
        note = (f"  (was {m['before']['train_rows']}, removed "
                f"{m['after']['rows_removed']} = {m['after']['removed_pct']}%)")
    print(f"{d:<32} train={n}{note}")

print("\nThe pre-fix dirs (data/splits_combined, data/splits_codipas_cls) still "
      "show a leak\nabove - that is expected and correct. They are kept unedited "
      "so older results\nstay traceable. Nothing below trains on them.")

## 2. Experiment 1 - Data / label audit

No training, no GPU, ~seconds. Run it first: it reports the class imbalance that
decides whether Experiments 3, 4 and 5 are even worth the GPU time.

Runs on the **Annotated** corpus - the labels we trust and the one every other
experiment is scored against.

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 1, "--splits", ANNOTATED, "--out", f"{OUT_ROOT}/exp1"],
    "E1 - data / label audit (Annotated)")

## 3. Experiment 7 - the flat multilabel headline

Run this **before** the heavier ablations. It is the number most likely to be
quoted, it trains on Annotated so home == yardstick, and it gives you a working
result early in case the session is cut short.

~3 seeds x 4 epochs. Budget roughly 40-60 min on a T4.

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 7, "--task", "multilabel", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--loss", "weighted_bce",
     "--out", f"{OUT_ROOT}/exp7"],
    "E7 - flat multilabel report (Annotated)")

In [ ]:
# Score every E7 checkpoint on both exams. Trained on Annotated, so home ==
# yardstick here and the script evaluates once and says so - the row still
# lands in two_exams.csv, which keeps the table shape identical across
# experiments.
for ck in sorted(Path(f"{OUT_ROOT}/exp7/checkpoints").glob("*")):
    if ck.is_dir():
        run([PY, "-m", "src.eval_two_exams", "--checkpoint", ck,
             "--out", f"{OUT_ROOT}/exp7", "--max-labels", 0, "--tag", ck.name])

display(pd.read_csv(f"{OUT_ROOT}/exp7/two_exams.csv").round(3))

## 4. Experiment 2 - Dataset ablation (the big one)

**The question:** does adding CODIPAS help, hurt, or do nothing?

**What we expect:** it hurts the fine-grained task. `docs/codipas_agreement.md`
measured the two label sets over the 2,520 shared texts:

| comparison | agreement | Cohen's κ |
|---|---|---|
| binary - distorted or not? | 66.5% | 0.321 (fair) |
| 11-class - which distortion? | 36.8% | **0.199 (slight)** |
| which type, among rows *both* call distorted | **27.5%** | — |

Even where both agree a distortion is present, they disagree about **which one
73% of the time**. Two schemes agreeing at κ = 0.199 are not labelling the same
thing, so this is not a "more data" experiment - it asks what merging
contradictory labels costs you.

**A clear negative result on a fixed test set is a real finding**, and more
defensible than a marginal win. That is why it is worth the GPU hours.

Each cell trains one corpus (all 3 seeds) and frees the GPU before the next.
Run them in order, or across separate sessions - results accumulate in the same
CSV. Budget ~1 hour per corpus.

In [ ]:
# 4a. Annotated only - the baseline arm. home == yardstick.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel", "--only-config", "annotated_only",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN, "--out", f"{OUT_ROOT}/exp2"],
    "E2a - Annotated only")

In [ ]:
# 4b. CODIPAS only (clean). This is the transfer arm: home is the CODIPAS test
# set, yardstick is Annotated, and the gap between them is the headline.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel", "--only-config", "codipas_only",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN, "--out", f"{OUT_ROOT}/exp2"],
    "E2b - CODIPAS only (clean)")

In [ ]:
# 4c. Annotated + CODIPAS (clean). Same val/test as Annotated, so home ==
# yardstick; only the training pool grew.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel",
     "--only-config", "annotated_plus_codipas",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN, "--out", f"{OUT_ROOT}/exp2"],
    "E2c - Annotated + CODIPAS (clean)")

In [ ]:
# 4d. Score all three arms on both exams. THIS is what makes E2 readable:
# without the yardstick column, the three arms sat three different exams.
for ck in sorted(Path(f"{OUT_ROOT}/exp2/checkpoints").glob("*")):
    if ck.is_dir():
        run([PY, "-m", "src.eval_two_exams", "--checkpoint", ck,
             "--out", f"{OUT_ROOT}/exp2", "--max-labels", 0, "--tag", ck.name])

e2 = pd.read_csv(f"{OUT_ROOT}/exp2/two_exams.csv")
display(e2.round(3))

# The comparison the experiment exists for: same exam, three training corpora.
yard = e2[e2["exam"] == "yardstick"]
if not yard.empty:
    print("\nYARDSTICK ONLY (data/splits/test.csv - the same 253 rows for all "
          "three arms):")
    print(yard.groupby("trained_on")[["macro_f1", "micro_f1", "weighted_f1"]]
              .agg(["mean", "std"]).round(3).to_string())
    print("\nIf Annotated-only >= Combined, merging the corpora costs you "
          "accuracy -\nwhich is the predicted result, and a reportable one.")

## 5. Experiments 3 → 4 → 5 - the imbalance chain

**Run in order.** Each reads the previous winner: E4 is only worth running if
E3's weighted CE did not close the gap, and E5 pairs sampling with whichever
loss won.

All three train on **Annotated**, so home == yardstick and they compare directly
to each other, to E7, and to the Month-1 baselines.

Skip this whole section if E1's audit showed the imbalance is mild - that is
what E1 is for.

In [ ]:
# E3 - plain CE vs class-weighted CE (multiclass, 11 classes)
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 3, "--task", "multiclass", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--out", f"{OUT_ROOT}/exp3"],
    "E3 - CE vs weighted CE (Annotated, multiclass)")

In [ ]:
# E4 - focal vs class-balanced. Only run if E3's gain looks insufficient.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 4, "--task", "multiclass", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--gamma", 2.0, "--cb-beta", 0.999,
     "--out", f"{OUT_ROOT}/exp4"],
    "E4 - focal vs class-balanced (Annotated, multiclass)")

In [ ]:
# E5 - weighted sampling vs the best loss from E3/E4.
BEST_LOSS = "weighted_ce"      # <-- set from the E3/E4 tables above

run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 5, "--task", "multiclass", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--best-loss", BEST_LOSS,
     "--out", f"{OUT_ROOT}/exp5"],
    f"E5 - weighted sampling vs {BEST_LOSS} (Annotated, multiclass)")

## 6. Experiment 6 - per-label performance and thresholds

Which of the ten labels the model actually fails on, and whether per-label
thresholds help. Thresholds are swept on **val only** and then frozen for test -
tuning them on test would make the test number meaningless.

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 6, "--task", "multilabel", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--loss", "weighted_bce",
     "--out", f"{OUT_ROOT}/exp6"],
    "E6 - multilabel per-label + thresholds (Annotated)")

## 7. Experiment 8 - sequence length / truncation

Cheap. Reports how many rows are cut at 512 tokens and only launches a
Longformer comparison if the rate justifies it (`--run-longformer`).

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 8, "--splits", ANNOTATED, "--model", MODEL,
     "--max-length", MAX_LEN, "--truncation-threshold-pct", 10,
     "--out", f"{OUT_ROOT}/exp8"],
    "E8 - truncation analysis (Annotated)")

## 8. One comparable table

Collects every `two_exams.csv` under `results_rerun/` into a single table, and
applies the comparability rule: **yardstick exam, same task, `leaked = False`**.
Anything failing that is shown separately rather than silently mixed in.

In [ ]:
frames = []
for f in sorted(Path(OUT_ROOT).glob("exp*/two_exams.csv")):
    d = pd.read_csv(f)
    d.insert(0, "experiment", f.parent.name)
    frames.append(d)

if not frames:
    print("No two_exams.csv found yet - run the evaluation cells above.")
else:
    allr = pd.concat(frames, ignore_index=True)
    allr.to_csv(f"{OUT_ROOT}/all_runs_two_exams.csv", index=False)

    comparable = allr[(allr["exam"] == "yardstick") & (~allr["leaked"])]
    print(f"{len(comparable)} comparable rows of {len(allr)} total\n")

    for task, grp in comparable.groupby("task"):
        print(f"--- task: {task}  (yardstick = data/splits/test.csv) ---")
        headline = {"binary": "positive_class_f1",
                    "multiclass": "macro_f1_10"}.get(task, "macro_f1")
        piv = (grp.groupby(["experiment", "trained_on", "loss"])[headline]
                  .agg(["mean", "std", "count"]).round(3)
                  .sort_values("mean", ascending=False))
        piv.columns = [f"{headline}_mean", f"{headline}_std", "seeds"]
        display(piv)

    excluded = allr[(allr["exam"] != "yardstick") | (allr["leaked"])]
    if not excluded.empty:
        print(f"\n{len(excluded)} rows NOT in the comparison above "
              f"(home exam, or leaked). Shown for the transfer story only:")
        display(excluded[["experiment", "trained_on", "exam", "leaked",
                          "macro_f1", "micro_f1"]].round(3))

In [ ]:
# Transfer story: home vs yardstick for anything trained off-corpus.
allr = pd.read_csv(f"{OUT_ROOT}/all_runs_two_exams.csv")
tr = allr[~allr["home_is_yardstick"]]
if tr.empty:
    print("Nothing trained off-corpus yet - run E2b (CODIPAS only).")
else:
    p = tr.pivot_table(index=["experiment", "trained_on", "seed"],
                       columns="exam", values="macro_f1")
    p["transfer_gap"] = p["home"] - p["yardstick"]
    display(p.round(3))
    print("\nhome      = did it learn its own corpus?")
    print("yardstick = does that carry over to the human-annotated task?")
    print("gap       = how much was corpus-specific. Large positive gap means "
          "it learned\n            CODIPAS's conventions, not the task.")

## 9. Collect and zip

Copies everything into `/kaggle/working/` and packs one archive. Checkpoints are
excluded - each is ~500 MB and there are many.

**Run this before the session ends.** Kaggle reclaims the working directory on
teardown; anything not copied out is gone.

In [ ]:
import shutil, zipfile, datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dest = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT
zip_path = dest / f"empowerlens_rerun_{stamp}.zip"

def add(zf, path, arc_root):
    path = Path(path)
    if path.is_file():
        zf.write(path, Path(arc_root) / path.name)
    elif path.is_dir():
        for f in sorted(path.rglob("*")):
            # Weights are huge and reproducible from the code + splits.
            if f.is_file() and "checkpoints" not in f.parts:
                zf.write(f, Path(arc_root) / f.relative_to(path))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    add(zf, OUT_ROOT, OUT_ROOT)
    add(zf, "docs/RERUN_PLAN.md", "docs")
    # The clean manifests travel with the results: they record exactly what was
    # removed from training, so any number here can be traced to its data.
    for d in (CODIPAS, COMBINED, YARDSTICK):
        for name in ("clean_manifest.json", "split_manifest.json"):
            if (Path(d) / name).exists():
                add(zf, Path(d) / name, d)

if dest.name == "working":
    shutil.copytree(OUT_ROOT, dest / OUT_ROOT, dirs_exist_ok=True)

print(f"{zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
with zipfile.ZipFile(zip_path) as zf:
    print(f"{len(zf.namelist())} files")
if dest.name == "working":
    print("Find it in the Output pane on the right.")

## What you can and cannot claim afterwards

**Comparable** - two numbers may sit in the same column only if both are:

1. from the **yardstick** exam (`data/splits/test.csv`, the same 253 rows),
2. on the **same task** (2, 11 and 10 classes are three different exams),
3. marked `leaked = False`.

All three fields are on every row of `all_runs_two_exams.csv`, so this is
checkable rather than remembered.

**Not comparable, by design:**

- home-exam scores from different corpora - CODIPAS's test set is a different
  exam from Annotated's,
- val against test,
- anything in the old `results_experiments/` or `results_combined/`, which came
  from the contaminated splits.

**The two claims this notebook supports:**

- *Within-dataset* - "trained and tested on X, we get N" (the **home** row).
- *Transfer* - "trained on X, tested on the human-annotated set, we get M"
  (the **yardstick** row). `home - yardstick` is how much was corpus-specific.